# v18 — Pairwise Bradley-Terry Temporal Ranking

**설계 이력**: 1차 시도(프레임 4개 절대순위를 독립적으로 물어본 뒤 헝가리안 알고리즘으로 배정)는
수학적으로는 문제가 없었으나, 태스크 적합성 검토 결과 결함이 발견되어 폐기하고 아래로 재설계함
(자세한 논증은 대화 로그 참고 — 핵심만 요약하면 아래 '왜 쌍별 비교인가' 참고).

**핵심 설계**: 4프레임을 항상 원본 파일 순서로 **전부** 보여줘서 global narrative를 보존하되
(`idea.md` §8에서 기각된 순수 pairwise 추론과 다른 점), 매번 두 프레임만 골라 "Frame i가 Frame j보다
먼저 일어나는가?"를 **Yes/No로만** 묻는다 — 이건 v1부터 이 프로젝트 전체가 검증해온
가장 오래되고 가장 안정적인 메커니즘(log P(Yes) − log P(No))을 그대로 재사용하는 것이라 새로운
실패 지점이 거의 없음. 4C2=6개 쌍 전부에 대해 이렇게 얻은 로그오즈를 반대칭 행렬로 모으고,
행 평균을 내면(아래 수학 참고) **닫힌 형태 최소제곱 해**로 4개 프레임의 전역 순위가 바로 나온다
(헝가리안 같은 이산 최적화조차 불필요 — 실수 4개를 정렬하면 끝).

### 왜 쌍별 비교인가 (태스크 적합성)
이 태스크의 실제 시각적 증거(인접 프레임 시각적 유사도, `idea.md` §6/§5-1)는 본질적으로
**비교적**이다 — "이 프레임이 저 프레임보다 사건이 더 진행됐다" 같은 상대판단이지, 프레임 하나만
떼어놓고 절대적 위치를 판단하는 게 아님. v1~v17 전부(성공/실패 불문) "프레임 배열 전체를 놓고
판단"하는 프레이밍이었다는 공통점이 있고, 1차 설계(절대순위 독립판단)는 이 프레이밍을 깨서
`idea.md` §8에서 이미 기각됐던 순수 pairwise 추론과 같은 종류의 손실(비교 불가능)을 겪었음.
쌍별 비교 + 4프레임 전체 맥락 유지로 이 문제를 해결.

### 수학: Bradley-Terry + Hodge 분해의 닫힌 형태 해
쌍 $(i,j)$, $i<j$마다 $z_{ij} = \log P(\text{Yes}) - \log P(\text{No})$("i가 j보다 먼저?")를 얻으면,
이건 Bradley-Terry 모델의 $s_i - s_j$ 추정치다($P(i<j)=\sigma(s_i-s_j)$). $K_4$(완전그래프, 각 변
관측치 1개)에서 $\sum_{i<j}(s_i-s_j-z_{ij})^2$를 최소화하는 최소제곱 해는, $z_{ji}=-z_{ij}$로
반대칭 완성한 뒤 **행 평균을 내는 것과 정확히 같다**(그래프 라플라시안 $L=nI-J$의 구조상
$Jb=0$이 자동으로 성립해서 $\hat s = b/n$이 $Ls=b$를 정확히 만족 — 유도는 대화 로그 참고).
즉 별도 solver 없이 `Z.mean(axis=1)`이 곧 정답이다. 학습 손실(쌍별 BCE)과 추론(행 평균)이
똑같은 확률모델(Bradley-Terry)에서 나온 것이라 이론과 구현이 어긋나지 않음.

### d=1 직접 타겟팅
정답 순위상 인접한(Kendall distance=1) 쌍은 `ADJACENT_PAIR_WEIGHT`로 손실에 더 큰 가중치를 줌 —
`idea.md` §5-1의 가장 완고한 약점을 손실 함수 차원이 아니라 **질문 자체를 그 쌍에 정확히 겨냥**
해서 공략(AdaptiveDistanceLoss의 온도 튜닝(§5-5), vision encoder 추가학습(v11/v12) 둘 다 이 비율을
못 줄였다는 게 이미 확인됐으므로, 이것도 확정적 해법이 아니라 실험 대상임을 명시).

### 리스크 (정직하게 명시)
- v14 대비 완전히 새 아키텍처 — 5-4절 단일변수 원칙이 그대로 적용 안 됨, v14는 그대로 안전한
  제출 후보로 유지
- "쌍별 비교가 실제로 학습 가능한 신호인가"는 이론이 아니라 실측으로만 확인 가능(§9 자가진단
  전에 소규모 스모크 테스트 권장)
- val↔LB 상관관계가 이 프로젝트에서 신뢰성이 낮았던 전례가 세 번 있음(`idea.md` §4-1) — 스모크
  테스트가 좋게 나와도 확정적 신호는 아님

### 효율
샘플당 forward: 학습·추론 공통 **6회**(v14의 8회보다 적고, v1~v17의 24회 전수조사 대비 4배
절감 — RTX 3090 단일 GPU 24시간 검증 리스크(`idea.md` §7 1순위)에 직접 도움).

## 1. 환경 체크

In [ ]:
import sys, os, subprocess, importlib
import pandas as pd

# 최신 transformers가 이 서버 torch 버전엔 없는 torch.distributed.tensor.DTensor를 내부적으로
# 요구해서 import 자체가 깨지는 경우가 있음(실제로 여러 번 재현됨) — Qwen3-VL은 지원하면서
# 이 DTensor 리팩터링 이전의 안정 버전으로 항상 고정. importlib 체크보다 먼저 실행해서
# '이미 설치돼 있으니 건너뜀' 판정에 걸리지 않게 함(bare import는 성공해도 AutoProcessor
# 접근 시점에서야 깨지는 lazy-import라 REQUIRED 체크만으론 못 잡음).
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.1"], check=True)
print("transformers==4.57.1 로 고정 설치 완료 (DTensor import 충돌 방지)")

REQUIRED = ["torch", "transformers", "peft", "accelerate", "pandas", "PIL", "tqdm", "huggingface_hub", "kagglehub"]
missing = []
for pkg in REQUIRED:
    name = "PIL" if pkg == "PIL" else pkg
    try:
        importlib.import_module(name)
    except ImportError:
        missing.append(pkg)

if missing:
    print(f"누락된 패키지 설치 중: {missing}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + missing, check=True)
else:
    print("필요한 패키지 전부 설치되어 있음")

import torch
print(f"torch={torch.__version__}  cuda_available={torch.cuda.is_available()}")

# FlashAttention2 — 멀티이미지 어텐션 특성상 sdpa/eager 대비 속도 차이가 큼.
# 컴파일이 필요해 수 분 걸릴 수 있고 환경에 따라 실패할 수 있음 — 실패해도 sdpa로 자동 폴백되니 여기서 안 죽음.
try:
    import flash_attn
    print(f"flash-attn 이미 설치됨: {flash_attn.__version__}")
except ImportError:
    print("flash-attn 설치 시도 중... (컴파일 필요, 수 분 소요될 수 있음)")
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "flash-attn", "--no-build-isolation"], check=True)
        import flash_attn
        print(f"flash-attn 설치 완료: {flash_attn.__version__}")
    except Exception as e:
        print(f"flash-attn 설치 실패 — sdpa로 폴백함(속도 저하 있을 수 있음): {e}")

## 2. Config

이 서버에서만 다르게 잡아야 하는 값은 이 셀만 수정하면 된다. **다른 v16/v17/v18 폴더와 모델·
데이터를 이미 받아둔 서버라면, 재다운로드 대신 심볼릭 링크로 재사용 권장**
(`ln -s ../v16_curriculum/models ./v18_rank_assignment/models` 등).

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path(".")
DATA_DIR     = Path("./data/snuaichallenge_data")
MODEL_LOCAL  = Path("./models/Qwen3-VL-8B-Instruct")
MODEL_HF_ID  = "Qwen/Qwen3-VL-8B-Instruct"  # 로컬에 없으면 여기서 다운로드
CKPT_DIR     = PROJECT_ROOT / "checkpoints"
LOG_DIR      = PROJECT_ROOT / "logs"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

MAX_IMAGE_SIZE = 448
VAL_RATIO      = 0.05
SEED           = 42

# LoRA는 v14와 100% 동일(검증된 값 재사용, 이 축은 안 건드림)
LORA_R, LORA_ALPHA, LORA_DROPOUT = 128, 256, 0.05
LR = 5e-5
EPOCHS = 5
BATCH_SIZE = 1          # DataLoader 배치 단위 = 샘플 1개 = 쌍별 질문 6개(그룹 크기 6)
GRAD_ACCUM = 8
WARMUP_RATIO = 0.05
LOGGING_STEPS = 50
# 그룹 크기가 v14의 8 -> 6으로 줄어서 TRAIN_MINIBATCH를 더 크게 잡을 여지가 있을 수 있음 —
# 본 학습 전에 §9 VRAM 자가진단으로 이 서버 기준 최댓값을 먼저 확인할 것. 아래는 보수적 시작값.
TRAIN_MINIBATCH  = 16
INFER_BATCH_SIZE = 16

# d=1(인접) 쌍에 손실 가중치를 더 줘서 이 프로젝트의 가장 완고한 약점(idea.md 5-1절)을 직접 타겟팅
# — 단, 확정적 해법이 아니라 실험 대상(과거 loss reweighting 시도들이 d=1 비율을 못 줄인 전례 있음)
ADJACENT_PAIR_WEIGHT = 1.5

CKPT_NAME = "best_v18"
print("config 로드 완료 (pairwise Bradley-Terry, group_size=6)")

## 3. GPU 개수 자동 감지

In [ ]:
import torch
NUM_GPUS = torch.cuda.device_count()
if NUM_GPUS == 0:
    raise RuntimeError("GPU가 감지되지 않았습니다.")
print(f"감지된 GPU 개수: {NUM_GPUS}")
for i in range(NUM_GPUS):
    props = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i} = {props.name}  VRAM={props.total_memory/1e9:.1f}GB")

## 4. 모델 가중치 확인/다운로드

로컬(`./models/...`)에 이미 완전하게 있으면 그대로 쓰고, 없으면 `!hf download` 셸 명령으로
정확히 `./` 밑에만 받는다 — `HF_HOME`도 `./`로 강제 리다이렉트해서 root 홈 디렉토리는 전혀
안 건드림 (Qwen3-VL-8B-Instruct는 공개 모델, gated 아님).

In [ ]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HOME"] = str((PROJECT_ROOT / ".cache" / "huggingface").resolve())
HF_TOKEN = "YOUR_HF_TOKEN_HERE"
os.environ["HF_TOKEN"] = HF_TOKEN

def model_is_complete():
    index_file = MODEL_LOCAL / "model.safetensors.index.json"
    if not index_file.exists():
        return False
    import json
    weight_map = json.loads(index_file.read_text())["weight_map"]
    return all((MODEL_LOCAL / f).exists() for f in set(weight_map.values()))

if model_is_complete():
    print(f"모델 이미 존재(무결성 확인됨): {MODEL_LOCAL}")
else:
    MODEL_LOCAL.parent.mkdir(parents=True, exist_ok=True)
    _stale_locks = list(MODEL_LOCAL.rglob("*.lock"))
    if _stale_locks:
        print(f"stale lock {len(_stale_locks)}개 정리: {[str(p) for p in _stale_locks]}")
        for _lf in _stale_locks:
            _lf.unlink(missing_ok=True)
    print(f"모델 다운로드 중 -> {MODEL_LOCAL} (hf download, ./ 밑에 직접 설치)")
    !hf download {MODEL_HF_ID} --local-dir {MODEL_LOCAL} --token {HF_TOKEN}
    if not model_is_complete():
        raise RuntimeError(f"다운로드 후에도 {MODEL_LOCAL}의 safetensors 샤드가 불완전합니다 — 위 로그 확인 필요")
    print(f"다운로드 완료: {MODEL_LOCAL}")

MODEL_PATH = str(MODEL_LOCAL)

## 5. 데이터 확인/다운로드 (Kaggle API)

Kaggle 토큰은 코드 안에 직접 박아두고, 실행 시점에 표준 경로 `~/.kaggle/access_token`에
자동으로 기록해서 `kagglehub`가 인증하도록 한다. `KAGGLEHUB_CACHE`와 `output_dir`을 둘 다
`./` 밑으로 강제해서 root 홈 디렉토리는 절대 안 씀.

In [ ]:
KAGGLE_TOKEN = "YOUR_KAGGLE_API_TOKEN_HERE"
_kdir = Path.home() / ".kaggle"
_kdir.mkdir(parents=True, exist_ok=True)
_ktok = _kdir / "access_token"
_ktok.write_text(KAGGLE_TOKEN)
os.chmod(_ktok, 0o600)
os.environ["KAGGLEHUB_CACHE"] = str((PROJECT_ROOT / ".cache" / "kagglehub").resolve())

if (DATA_DIR / "train.csv").exists() and (DATA_DIR / "test.csv").exists():
    print(f"데이터 이미 존재: {DATA_DIR}")
else:
    import shutil
    # kagglehub는 output_dir이 이미 존재하고 비어있지 않으면 FileExistsError를 던짐
    # (이전에 중단된 다운로드가 남긴 부분 파일이 있으면 여기 걸림) — 서브프로세스라 이 에러가
    # 노트북 커널로 전파 안 되고 조용히 넘어가서 아래 rglob도 실패 -> 'no file' 증상으로 나타남.
    # 그래서 매번 깨끗하게 비우고 시작 + force_download=True로 이중 안전장치.
    if DATA_DIR.exists():
        print(f"기존 불완전 데이터 폴더 정리: {DATA_DIR}")
        shutil.rmtree(DATA_DIR)
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print(f"데이터 다운로드 중 -> {DATA_DIR} (kagglehub, ./ 밑에 직접 설치)")
    !python3 -c "import kagglehub; kagglehub.competition_download('snuaichallenge', output_dir='{DATA_DIR}', force_download=True)"
    if not (DATA_DIR / "train.csv").exists():
        candidates = list(DATA_DIR.rglob("train.csv"))
        if candidates:
            DATA_DIR = candidates[0].parent
        else:
            raise FileNotFoundError(f"{DATA_DIR} 아래에서 train.csv를 못 찾았습니다 — 위 다운로드 로그를 확인하세요.")
    print(f"최종 DATA_DIR = {DATA_DIR}")

## 6. Pairwise Dataset / Model 정의

`Answer[i-1]`은 원본 파일순서 프레임 i의 정답 시간위치(1~4) — 두 프레임의 순위를 비교하면
바로 "먼저/나중" 라벨이 나온다. Yes/No 로그오즈 읽기는 v1부터 검증된 메커니즘을 100% 그대로
재사용(새 토큰, 새 헤드 전혀 없음).

In [ ]:
import ast, time
from itertools import combinations
from PIL import Image
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

PAIRS = list(combinations(range(1, 5), 2))  # [(1,2),(1,3),(1,4),(2,3),(2,4),(3,4)]

PROMPT_PAIR = (
    "Sentence: {sentence}\n\n"
    "These 4 frames are shown in their original file order, which may NOT be the chronological order.\n"
    "Carefully examine the visual changes across all 4 frames together with the sentence.\n"
    "Does the event shown in Frame {i} happen before the event shown in Frame {j}, "
    "in the correct chronological order?\n"
    "Answer only with \"Yes\" or \"No\"."
)
SYSTEM = (
    "You are a temporal ordering assistant. "
    "Given 4 video frames shown in an arbitrary fixed order and a caption describing the full event, "
    "determine the relative chronological order between two specified frames."
)

def load_image(path):
    img = Image.open(path).convert("RGB")
    w, h = img.size
    scale = MAX_IMAGE_SIZE / max(w, h)
    if scale < 1.0:
        img = img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
    return img

def build_messages_pair(base_imgs, sentence, i, j):
    content = []
    for k, img in enumerate(base_imgs, 1):
        content.append({"type": "text", "text": f"Frame {k}:"})
        content.append({"type": "image", "image": img})
    content.append({"type": "text", "text": PROMPT_PAIR.format(sentence=sentence, i=i, j=j)})
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": content}]

class PairwiseTemporalDataset(Dataset):
    """샘플 1개 -> 쌍별 질문 6개(4C2). 항상 원본 파일 순서로 4프레임 전부를 보여주고,
    두 프레임 중 어느 게 먼저인지만 Yes/No로 묻는다."""
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        sid = row["Id"]
        ranks = ast.literal_eval(row["Answer"])  # ranks[k-1] = 프레임 k의 정답 시간위치(1~4)
        img_dir = DATA_DIR / "train" / sid
        files = sorted(f.name for f in img_dir.iterdir() if f.suffix == ".jpg")
        base_imgs = [load_image(str(img_dir / f)) for f in files]
        images_list, sentences, pair_ijs, labels, adj_flags = [], [], [], [], []
        for (i, j) in PAIRS:
            images_list.append(base_imgs)
            sentences.append(row["Sentence"])
            pair_ijs.append((i, j))
            labels.append(1.0 if ranks[i - 1] < ranks[j - 1] else 0.0)
            adj_flags.append(1 if abs(ranks[i - 1] - ranks[j - 1]) == 1 else 0)
        return {"sid": sid, "images": images_list, "sentences": sentences, "pair_ijs": pair_ijs,
                "labels": labels, "adj_flags": adj_flags, "group_size": len(PAIRS)}

def collate_fn(batch):
    return {
        "sids": [b["sid"] for b in batch],
        "images": [b["images"] for b in batch],
        "sentences": [b["sentences"] for b in batch],
        "pair_ijs": [b["pair_ijs"] for b in batch],
        "labels": [b["labels"] for b in batch],
        "adj_flags": [b["adj_flags"] for b in batch],
        "group_sizes": [b["group_size"] for b in batch],
    }

def get_model_class(model_path):
    from transformers import AutoConfig
    cfg = AutoConfig.from_pretrained(model_path)
    mt = getattr(cfg, "model_type", "")
    if mt == "qwen3_vl":
        from transformers import Qwen3VLForConditionalGeneration
        return Qwen3VLForConditionalGeneration
    if mt == "qwen2_5_vl":
        from transformers import Qwen2_5_VLForConditionalGeneration
        return Qwen2_5_VLForConditionalGeneration
    from transformers import Qwen2VLForConditionalGeneration
    return Qwen2VLForConditionalGeneration

def load_model_and_processor(resume_from=None):
    from transformers import AutoProcessor
    from peft import LoraConfig, PeftModel, get_peft_model, TaskType
    processor = AutoProcessor.from_pretrained(MODEL_PATH)
    ModelClass = get_model_class(MODEL_PATH)
    model = None
    _errs = []
    for attn_impl in ("flash_attention_2", "sdpa", "eager"):
        try:
            model = ModelClass.from_pretrained(MODEL_PATH, torch_dtype=torch.bfloat16, attn_implementation=attn_impl)
            break
        except Exception as e:
            _errs.append(f"{attn_impl}: {e}")
    if model is None:
        raise RuntimeError("모델 로드 3가지 attn_implementation 전부 실패:\n" + "\n".join(_errs))
    model.gradient_checkpointing_enable()
    if resume_from:
        model = PeftModel.from_pretrained(model, resume_from, is_trainable=True)
    else:
        lora_config = LoraConfig(
            task_type=TaskType.CAUSAL_LM, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
            bias="none",
        )
        model = get_peft_model(model, lora_config)
    return model, processor

def get_yes_no_token_ids(processor):
    tok = processor.tokenizer
    return tok.convert_tokens_to_ids(tok.tokenize("Yes"))[-1], tok.convert_tokens_to_ids(tok.tokenize("No"))[-1]

def forward_logit(model, inputs, yes_id, no_id):
    outputs = model(**inputs)
    last_logits = outputs.logits[:, -1, :].float()
    log_probs = torch.log_softmax(last_logits, dim=-1)
    score = log_probs[:, yes_id] - log_probs[:, no_id]
    return score.clamp(-100.0, 100.0)

def pairwise_bt_loss(logits, labels, adj_flags):
    """Bradley-Terry negative log-likelihood — BCE(z_ij, y_ij)의 평균. d=1(인접) 쌍은 가중치 상향.
    reduction='mean'이라 그룹 크기(6)에 무관하게 손실 스케일이 v14의 단일 8-way softmax 항과
    비슷한 크기로 유지됨 -> LR=5e-5를 그대로 재사용해도 무리가 없음."""
    labels_t = torch.tensor(labels, device=logits.device, dtype=logits.dtype)
    weights = torch.tensor([ADJACENT_PAIR_WEIGHT if a else 1.0 for a in adj_flags],
                            device=logits.device, dtype=logits.dtype)
    return F.binary_cross_entropy_with_logits(logits, labels_t, weight=weights, reduction="mean")

print("핵심 로직 정의 완료 (pairwise dataset / model / Bradley-Terry loss)")

## 7. 쌍별 로그오즈 -> 전역 순위 집계 (검증/추론 공용)

`score_pairs`(순수 forward, raw 로그오즈 6개 반환)와 `aggregate_ranks`(닫힌 형태 집계)를
**분리**해뒀다 — 학습 손실은 raw 로그오즈만 쓰고 집계 로직은 절대 안 건드리며(§10 train_fn),
검증/추론에서만 둘을 합쳐 쓴다. 이렇게 분리해두면 "쌍별 판단 자체가 맞았는가"(raw)와
"집계 알고리즘이 최종 순서를 얼마나 잘 복원했는가"(reconstruction)를 §8에서 따로 진단할 수 있다.

6개 쌍의 로그오즈를 반대칭 행렬로 모으고 행 평균을 내면, $K_4$(완전그래프·균형 관측)에서는
이게 정확히 최소제곱(Hodge 분해) 해와 같다 — 별도 solver 없이 `mean`만으로 충분
(유도는 위 §0 수학 설명 참고). 결과가 그대로 `Answer` 포맷.

In [ ]:
def score_pairs(model, processor, yes_id, no_id, base_imgs, sentence, minibatch):
    """순수 forward만 — 집계 로직과 완전히 분리. {(i,j): z_ij} 반환."""
    texts, imgs_list = [], []
    for (i, j) in PAIRS:
        msg = build_messages_pair(base_imgs, sentence, i, j)
        text = processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
        texts.append(text)
        imgs_list.append(base_imgs)
    zs = []
    with torch.no_grad():
        for bi in range(0, len(texts), minibatch):
            inp = processor(text=texts[bi:bi+minibatch], images=imgs_list[bi:bi+minibatch],
                             return_tensors="pt", padding=True).to(model.device)
            s = forward_logit(model, inp, yes_id, no_id)
            zs.extend(s.cpu().tolist())
    return dict(zip(PAIRS, zs))

def aggregate_ranks(pair_scores):
    """raw 로그오즈 dict -> 닫힌 형태 집계(행 평균) -> Answer 포맷 순위. 이산 최적화 없음."""
    Z = [[0.0] * 5 for _ in range(5)]  # 1-indexed, 반대칭
    for (i, j), z in pair_scores.items():
        Z[i][j] = z
        Z[j][i] = -z
    s = [sum(Z[i][j] for j in range(1, 5) if j != i) / 3.0 for i in range(1, 5)]
    order = sorted(range(1, 5), key=lambda i: -s[i - 1])  # s 높을수록(더 자주 '먼저') 이른 프레임
    pred_ranks = [0] * 4
    for rank_pos, frame_i in enumerate(order, 1):
        pred_ranks[frame_i - 1] = rank_pos
    return pred_ranks

def predict_permutation(model, processor, yes_id, no_id, base_imgs, sentence, minibatch):
    """aggregate_ranks 합친 편의 함수 — run_inference에서 사용.
    §7-1 검증 통과 후에는 score_pairs_fast(고속 경로)를 씀 — 검증 전이면 score_pairs로 바꿀 것."""
    return aggregate_ranks(score_pairs_fast(model, processor, yes_id, no_id, base_imgs, sentence))

print("score_pairs / aggregate_ranks / predict_permutation 정의 완료")

## 7-1. 고속 경로 — vision encoding 중복 제거 (실험적, 검증 필수)

**문제**: 위 `score_pairs`는 매번 `processor(text=[...6개...], images=[...같은 4장을 6번 반복...])`으로
6쌍을 배치 처리하는데, 이 배치 안에서 **동일한 4장의 이미지가 6번 반복**되어 vision encoder가
4장이 아니라 24장을 처리하는 셈이 됨 — 이게 스텝당 6초대로 느린 핵심 원인 중 하나.

**해법**: Qwen3-VL 내부 API(`get_image_features`, `get_placeholder_mask`, `get_rope_index`)를 직접 호출해서
vision encoder는 고유 이미지 4장에 대해 **딱 1번만** 돌리고, 그 결과(image_embeds + deepstack
embeds)를 6개 텍스트 시퀀스에 복제해서 LLM 레이어만 배치로 통과시킨다 — LLM의 KV 캐시 재사용이
아니라 **vision encoder 출력만 재사용**하는 것이라 `idea.md` 5-7절에서 막혔던 M-RoPE/KV캐시
비호환 문제와는 다른 지점(각 시퀀스는 여전히 자기 position_ids를 새로 계산함).

**⚠️ 이건 transformers 내부 비공개 API에 직접 접근하는 실험적 최적화라 subtle bug 위험이 있음
— 반드시 아래 검증 셀에서 기존(느리지만 검증된) `score_pairs`와 결과가 일치하는지 확인 후에만
학습/추론에 쓸 것.**

In [ ]:
def _unwrap_qwen_causal_lm(model):
    """PeftModel으로 감싸져 있으면 벗겨서 Qwen3VLForConditionalGeneration(lm_head 보유)을 반환."""
    m = model
    if hasattr(m, "base_model") and hasattr(m.base_model, "model"):
        m = m.base_model.model
    return m

def _pairwise_logits_fast(model, processor, base_imgs, sentence, device, cached_vision=None):
    """4장 이미지를 vision encoder에 1번만 태우고(또는 cached_vision으로 아예 스킵), 6개 쌍별
    질문은 LLM 레이어만 배치로 통과시켜 (6,) 로그오즈 텐서를 반환(PAIRS 순서). 미분 가능.
    cached_vision=(image_embeds_once, deepstack_image_embeds)를 주면 vision encoder forward
    자체를 건너뜀 — vision encoder는 LoRA로 학습 안 되는 고정 파트라 항상 유효한 캐시.
    ⚠️ apply_chat_template(tokenize=False)만으로는 이미지 placeholder가 확장 안 된 마커 1개로
    남아있고, 실제 확장(image_grid_thw만큼 반복)은 processor.__call__ 내부에서만 일어남
    (processing_qwen3_vl.py 직접 확인함) — 그래서 반드시 processor(text=..., images=...)를
    통째로 호출해서 input_ids를 만들어야 함(tokenizer만 따로 부르면 placeholder 개수가 안 맞아
    get_placeholder_mask에서 터지거나, 최악의 경우 조용히 틀어질 수 있음)."""
    causal_lm = _unwrap_qwen_causal_lm(model)
    qwen = causal_lm.model  # Qwen3VLModel (vision + language_model)
    n_rows = len(PAIRS)

    texts = [processor.apply_chat_template(build_messages_pair(base_imgs, sentence, i, j),
                                            tokenize=False, add_generation_prompt=True)
             for (i, j) in PAIRS]
    full_inp = processor(text=texts, images=[base_imgs] * n_rows, return_tensors="pt", padding=True)
    input_ids = full_inp["input_ids"].to(device)
    attention_mask = full_inp["attention_mask"].to(device)
    image_grid_thw = full_inp["image_grid_thw"].to(device)  # 24행(4장 x 6번 반복)

    if cached_vision is not None:
        image_embeds_once, deepstack_image_embeds = cached_vision
        image_embeds_once = image_embeds_once.to(device)
        deepstack_image_embeds = [d.to(device) for d in deepstack_image_embeds]
    else:
        img_out = processor.image_processor(images=base_imgs, return_tensors="pt")
        pixel_values = img_out["pixel_values"].to(device)
        unique_grid_thw = img_out["image_grid_thw"].to(device)
        image_embeds_tuple, deepstack_image_embeds = qwen.get_image_features(pixel_values, unique_grid_thw)
        image_embeds_once = torch.cat(image_embeds_tuple, dim=0)  # (4장의 총 이미지토큰수, hidden)

    inputs_embeds = qwen.get_input_embeddings()(input_ids)
    image_embeds_rep = image_embeds_once.repeat(n_rows, 1).to(inputs_embeds.dtype)
    image_mask, _ = qwen.get_placeholder_mask(input_ids, inputs_embeds=inputs_embeds,
                                                image_features=image_embeds_rep)
    inputs_embeds = inputs_embeds.masked_scatter(image_mask, image_embeds_rep)

    position_ids, _ = qwen.get_rope_index(input_ids, image_grid_thw, None, attention_mask=attention_mask)

    visual_pos_masks = image_mask[..., 0]
    deepstack_visual_embeds_rep = [d.repeat(n_rows, 1) for d in deepstack_image_embeds]

    outputs = qwen.language_model(
        input_ids=None,
        position_ids=position_ids,
        attention_mask=attention_mask,
        inputs_embeds=inputs_embeds,
        visual_pos_masks=visual_pos_masks,
        deepstack_visual_embeds=deepstack_visual_embeds_rep,
    )
    hidden = outputs.last_hidden_state  # (n_rows, seq, hidden)

    # 마지막 유효 토큰 위치(패딩 방향 무관하게 attention_mask로 안전하게 계산)
    seq_positions = torch.arange(input_ids.shape[1], device=device).unsqueeze(0).expand(n_rows, -1)
    masked_positions = torch.where(attention_mask.bool(), seq_positions, torch.full_like(seq_positions, -1))
    last_idx = masked_positions.max(dim=1).values
    last_hidden = hidden[torch.arange(n_rows, device=device), last_idx]  # (n_rows, hidden)

    last_logits = causal_lm.lm_head(last_hidden).float()
    log_probs = torch.log_softmax(last_logits, dim=-1)
    yes_id, no_id = get_yes_no_token_ids(processor)
    z = (log_probs[:, yes_id] - log_probs[:, no_id]).clamp(-100.0, 100.0)
    return z  # (6,), PAIRS 순서

def score_pairs_fast(model, processor, yes_id, no_id, base_imgs, sentence, minibatch=None, cached_vision=None):
    """score_pairs와 동일한 인터페이스({(i,j): z_ij})지만 고속 경로 사용. minibatch 인자는
    호출부 호환용(안 씀 — 6쌍이 항상 1번의 LLM 배치 forward로 처리됨)."""
    with torch.no_grad():
        z = _pairwise_logits_fast(model, processor, base_imgs, sentence, model.device, cached_vision)
    return dict(zip(PAIRS, z.cpu().tolist()))

print("고속 경로(_pairwise_logits_fast / score_pairs_fast) 정의 완료 — 아래 검증 셀 반드시 먼저 실행할 것")

### 7-1 검증 — 반드시 먼저 실행 (기존 느린 경로 + 캐시 경로 결과 일치 확인)

train.csv에서 실제 샘플 3개를 뽑아 `score_pairs`(느리지만 검증된) vs `score_pairs_fast`(캐시 없이,
vision encoder 즉석 계산) vs `score_pairs_fast`(캐시로 vision encoder 스킵) 세 가지를 전부 비교한다
(단위: 로그오즈 점수). 최대 오차가 1e-2(부동소수점 오차 범위) 안이면 통과 — 하나라도 어긋나면
`AssertionError`가 뜨니, 그러면 고속 경로를 쓰지 말고 알려줄 것.

In [ ]:
def verify_fast_path():
    model, processor = load_model_and_processor()
    model = model.to("cuda:0").eval()
    yes_id, no_id = get_yes_no_token_ids(processor)
    qwen = _unwrap_qwen_causal_lm(model).model

    train_csv = pd.read_csv(DATA_DIR / "train.csv")
    sample_rows = train_csv.sample(n=3, random_state=SEED)

    max_diff = 0.0
    for _, row in sample_rows.iterrows():
        sid, sentence = row["Id"], row["Sentence"]
        img_dir = DATA_DIR / "train" / sid
        files = sorted(f.name for f in img_dir.iterdir() if f.suffix == ".jpg")
        base_imgs = [load_image(str(img_dir / f)) for f in files]

        slow = score_pairs(model, processor, yes_id, no_id, base_imgs, sentence, 16)
        fast = score_pairs_fast(model, processor, yes_id, no_id, base_imgs, sentence)

        with torch.no_grad():
            img_out = processor.image_processor(images=base_imgs, return_tensors="pt")
            pv = img_out["pixel_values"].to("cuda:0")
            gthw = img_out["image_grid_thw"].to("cuda:0")
            emb_tuple, deepstack = qwen.get_image_features(pv, gthw)
            fake_cache = (torch.cat(emb_tuple, dim=0), deepstack)
        fast_cached = score_pairs_fast(model, processor, yes_id, no_id, base_imgs, sentence,
                                        cached_vision=fake_cache)

        for pair in PAIRS:
            d1 = abs(slow[pair] - fast[pair])
            d2 = abs(slow[pair] - fast_cached[pair])
            max_diff = max(max_diff, d1, d2)
            print(f"  {sid} {pair}  slow={slow[pair]:.4f}  fast={fast[pair]:.4f}  "
                  f"fast_cached={fast_cached[pair]:.4f}  diff={max(d1,d2):.5f}")

    print(f"\n최대 오차: {max_diff:.5f}")
    if max_diff > 1e-2:
        raise AssertionError(
            f"고속 경로 결과가 기존 경로와 {max_diff:.5f}만큼 어긋남 — 고속 경로에 버그가 있는 것으로 "
            "보임. 이 상태로 학습/추론에 쓰면 안 됨. score_pairs(느린 경로)로 되돌릴 것."
        )
    print("검증 통과 — 고속 경로(캐시 포함)를 학습/검증/추론에 안전하게 사용 가능")
    del model
    torch.cuda.empty_cache()

verify_fast_path()

## 7-2. Vision Embedding 캐시 사전 계산

**vision encoder는 LoRA로 학습되지 않는 완전 고정 파트**(`target_modules`가 LLM 디코더에만
붙음, `idea.md` 확인됨) — 그래서 같은 이미지의 vision embedding은 에폭이 몇 번을 돌든 항상
동일하다. train.csv 전체(9535장 세트)의 vision embedding을 학습 시작 전 **딱 한 번만**
계산해서 디스크에 캐시해두면, 학습 루프 내내 vision encoder forward 자체가 필요 없어진다
(5에폭 학습이면 기존 대비 vision 인코딩 비용이 5분의 1 이하로 줄어드는 효과 — §7-1에서
이미 없앤 '한 스텝 안 6배 중복'과는 별개의, 훨씬 큰 폭의 절감).

`./vision_cache/<Id>.pt`에 저장하고, 이미 있는 샘플은 건너뛰어서 재실행해도 안전함(중단 후
재개 가능). 저장 용량은 샘플당 대략 수 MB대로 추정 — 전체 9535개면 디스크에 수십 GB 필요할
수 있으니 `./` 볼륨(컨테이너 디스크 말고 마운트된 볼륨)에 여유 있는지 미리 확인할 것.

In [ ]:
VISION_CACHE_DIR = PROJECT_ROOT / "vision_cache"
VISION_CACHE_DIR.mkdir(parents=True, exist_ok=True)

def load_vision_cache(sid):
    path = VISION_CACHE_DIR / f"{sid}.pt"
    if not path.exists():
        return None
    return torch.load(path, map_location="cpu")

def precompute_vision_cache():
    model, processor = load_model_and_processor()
    model = model.to("cuda:0").eval()
    qwen = _unwrap_qwen_causal_lm(model).model

    train_csv = pd.read_csv(DATA_DIR / "train.csv")
    todo = [row for _, row in train_csv.iterrows() if not (VISION_CACHE_DIR / f"{row['Id']}.pt").exists()]
    print(f"캐시할 샘플: {len(todo)}/{len(train_csv)} (이미 있는 건 자동으로 건너뜀)")

    t0 = time.time()
    with torch.no_grad():
        for n, row in enumerate(todo):
            sid = row["Id"]
            img_dir = DATA_DIR / "train" / sid
            files = sorted(f.name for f in img_dir.iterdir() if f.suffix == ".jpg")
            base_imgs = [load_image(str(img_dir / f)) for f in files]

            img_out = processor.image_processor(images=base_imgs, return_tensors="pt")
            pixel_values = img_out["pixel_values"].to("cuda:0")
            image_grid_thw = img_out["image_grid_thw"].to("cuda:0")
            emb_tuple, deepstack = qwen.get_image_features(pixel_values, image_grid_thw)
            image_embeds_once = torch.cat(emb_tuple, dim=0).cpu()
            deepstack_cpu = [d.cpu() for d in deepstack]

            torch.save((image_embeds_once, deepstack_cpu), VISION_CACHE_DIR / f"{sid}.pt")
            if (n + 1) % 200 == 0 or (n + 1) == len(todo):
                elapsed = time.time() - t0
                rate = (n + 1) / elapsed
                eta_min = (len(todo) - (n + 1)) / rate / 60 if rate > 0 else 0
                print(f"  {n+1}/{len(todo)}  {elapsed/60:.1f}분 경과  ETA={eta_min:.1f}분")

    del model
    torch.cuda.empty_cache()
    print(f"캐시 완료: {VISION_CACHE_DIR}")

precompute_vision_cache()

## 8. Validation 함수

세 가지를 따로 본다:
1. **raw_pairwise_acc** — 집계 이전, 모델이 쌍별 질문에 직접 답한 정확도. 학습 신호 자체의
   품질을 순수하게 측정(집계 알고리즘의 보정 효과가 안 섞임)
2. **raw_adjacent_acc** — 위 중 정답 순위상 인접한(d=1) 쌍만 따로. `idea.md` 5-1절의 가장
   완고한 약점이 이번에도 안 풀리는지 직접 감시하는 지표(과거 loss reweighting들이 전부
   이 비율을 못 줄였던 전례가 있어, 여기서도 낮게 나올 가능성을 열어두고 봐야 함)
3. **exact_match / recon_pairwise_acc** — 6개 raw 판단을 집계한 최종 순열의 성능. raw보다
   높게 나오면 집계(과결정 구조)가 개별 오류를 보정해주고 있다는 뜻.

In [ ]:
def val_exact_match(model, processor, val_raw_df, yes_id, no_id, device):
    model.eval()
    correct = 0
    raw_correct, raw_total = 0, 0
    adj_correct, adj_total = 0, 0
    recon_correct, recon_total = 0, 0
    for _, row in val_raw_df.iterrows():
        sid, sentence = row["Id"], row["Sentence"]
        gt = ast.literal_eval(row["Answer"])
        img_dir = DATA_DIR / "train" / sid
        files = sorted(f.name for f in img_dir.iterdir() if f.suffix == ".jpg")
        base_imgs = [load_image(str(img_dir / f)) for f in files]

        pair_scores = score_pairs_fast(model, processor, yes_id, no_id, base_imgs, sentence,
                                        cached_vision=load_vision_cache(sid))
        for (i, j), z in pair_scores.items():
            pred_before = z > 0
            gt_before = gt[i - 1] < gt[j - 1]
            is_adj = abs(gt[i - 1] - gt[j - 1]) == 1
            raw_correct += int(pred_before == gt_before)
            raw_total += 1
            if is_adj:
                adj_correct += int(pred_before == gt_before)
                adj_total += 1

        pred = aggregate_ranks(pair_scores)
        if pred == gt:
            correct += 1
        for (i, j) in PAIRS:
            recon_correct += int((pred[i - 1] < pred[j - 1]) == (gt[i - 1] < gt[j - 1]))
            recon_total += 1

    acc = correct / len(val_raw_df)
    raw_acc = raw_correct / raw_total
    adj_acc = adj_correct / adj_total
    recon_acc = recon_correct / recon_total
    print(f"[Val] exact_match={acc:.4f} ({correct}/{len(val_raw_df)})")
    print(f"      raw_pairwise_acc={raw_acc:.4f}  raw_adjacent(d=1)_acc={adj_acc:.4f}  "
          f"recon_pairwise_acc={recon_acc:.4f}")
    if adj_acc < raw_acc - 0.05:
        print(f"      [주의] d=1 인접쌍 정확도가 전체 대비 {raw_acc - adj_acc:.4f} 낮음 — "
              f"과거 loss reweighting들이 못 풀었던 것과 같은 패턴일 수 있음")
    model.train()
    return acc

## 9. VRAM 자가진단 (본 학습 전 필수)

그룹 크기가 v14의 8에서 6으로 줄었으니 `TRAIN_MINIBATCH`를 §2의 보수적 시작값(16)보다 더
키울 여지가 있을 수 있다 — 이 셀로 이 서버에서 실제로 OOM 없이 도는 최댓값을 먼저 확인하고
필요하면 §2로 돌아가 `TRAIN_MINIBATCH`를 조정할 것. `accelerator.accumulate()` 패턴으로
실제 학습 루프와 동일하게(옵티마이저 상태 할당 포함) 검증한다(`idea.md` 5-11절 교훈 반영).
**동시에 이 셀은 소규모 학습 신호 스모크 테스트를 겸한다 — loss가 실제로 떨어지는지,
pairwise_acc가 chance(50%)보다 유의미하게 높아지는지 먼저 확인할 것.**

In [ ]:
def vram_selfcheck(n_cycles=3):
    """고속 경로(_pairwise_logits_fast) 사용 — 그룹(6쌍)을 항상 1번의 배치 LLM forward로
    처리하므로 minibatch 청킹 개념이 없음(vision encoder 중복 제거로 이미 그룹 전체가 가벼워짐)."""
    from torch.optim import AdamW
    from accelerate import Accelerator
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    accelerator = Accelerator(gradient_accumulation_steps=GRAD_ACCUM)
    print(f"[설정] 고속 경로  n_cycles={n_cycles}  GRAD_ACCUM={GRAD_ACCUM}")
    t_setup0 = time.time()
    model, processor = load_model_and_processor()
    model = model.to(accelerator.device)
    yes_id, no_id = get_yes_no_token_ids(processor)
    optimizer = AdamW(model.parameters(), lr=LR)
    model, optimizer = accelerator.prepare(model, optimizer)
    print(f"[설정] 모델 로드+prepare 완료: {time.time()-t_setup0:.1f}초")

    train_csv = pd.read_csv(DATA_DIR / "train.csv")
    ds = PairwiseTemporalDataset(train_csv.sample(n=min(64, len(train_csv)), random_state=SEED))
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
    dl = accelerator.prepare(dl)

    def mem_str():
        cur = torch.cuda.memory_allocated() / 1e9
        peak = torch.cuda.max_memory_allocated() / 1e9
        return f"cur={cur:.2f}GB peak={peak:.2f}GB"

    try:
        cycle, micro_step = 0, 0
        cycle_times = []
        t_cycle0 = time.time()
        for batch in dl:
            with accelerator.accumulate(model):
                unwrapped = accelerator.unwrap_model(model)
                logit_parts, labels, adj_flags = [], [], []
                t0 = time.time()
                for sid, grp_imgs, grp_sents, grp_labels, grp_adj in zip(
                        batch["sids"], batch["images"], batch["sentences"], batch["labels"], batch["adj_flags"]):
                    base_imgs, sentence = grp_imgs[0], grp_sents[0]
                    z = _pairwise_logits_fast(unwrapped, processor, base_imgs, sentence, accelerator.device,
                                               cached_vision=load_vision_cache(sid))
                    logit_parts.append(z)
                    labels.extend(grp_labels)
                    adj_flags.extend(grp_adj)
                torch.cuda.synchronize()
                micro_step += 1
                print(f"    [micro {micro_step}] (고속경로, 그룹 {len(logit_parts)}개)  "
                      f"{time.time()-t0:.2f}초  {mem_str()}")
                logits = torch.cat(logit_parts)
                loss = pairwise_bt_loss(logits, labels, adj_flags)
                accelerator.backward(loss)
                optimizer.step()
                optimizer.zero_grad()
            if accelerator.sync_gradients:
                cycle += 1
                t_cycle = time.time() - t_cycle0
                cycle_times.append(t_cycle)
                avg_cycle = sum(cycle_times) / len(cycle_times)
                eta = avg_cycle * (n_cycles - cycle)
                print(f"  cycle {cycle}/{n_cycles}  loss={loss.item():.4f}  {mem_str()}  "
                      f"cycle_time={t_cycle:.1f}초  avg={avg_cycle:.1f}초  ETA={eta:.1f}초")
                t_cycle0 = time.time()
                if cycle >= n_cycles:
                    break
        print(f"OK, peak={torch.cuda.max_memory_allocated()/1e9:.2f}GB, "
              f"평균 cycle={sum(cycle_times)/len(cycle_times):.1f}초")
        return True
    except torch.cuda.OutOfMemoryError:
        print(f"OOM (micro_step={micro_step}에서 실패, {mem_str()})")
        return False
    finally:
        del model, optimizer
        torch.cuda.empty_cache()

# OOM나면 반드시 커널 재시작 후 재시도(캐싱 할당자 파편화로 재시도가 더 빨리 OOM나는 경우가 흔함).
# 고속 경로는 vision 중복 인코딩을 없애서 이미 훨씬 가벼움 — 그래도 OOM나면 §7-1을 다시
# 확인하거나(검증 통과했는지) 원래 score_pairs/forward_logit 경로로 되돌릴 것.
vram_selfcheck()

## 10. 학습 함수 (`accelerate.notebook_launcher`로 멀티 GPU 실행)

K=7 하드네거티브 샘플링이 없어서 v14/v16 train_fn보다 단순함 — 그룹 안 6개 쌍에 대해
Bradley-Terry BCE만 계산.

In [ ]:
def train_fn():
    from datetime import timedelta
    from torch.optim import AdamW
    from transformers import get_cosine_schedule_with_warmup
    from accelerate import Accelerator
    from accelerate.utils import InitProcessGroupKwargs

    pg_kwargs = InitProcessGroupKwargs(timeout=timedelta(days=2))
    accelerator = Accelerator(gradient_accumulation_steps=GRAD_ACCUM, kwargs_handlers=[pg_kwargs])
    device = accelerator.device
    is_main = accelerator.is_main_process
    torch.manual_seed(SEED)

    train_csv_full = pd.read_csv(DATA_DIR / "train.csv").sample(frac=1, random_state=SEED).reset_index(drop=True)
    n_val = int(len(train_csv_full) * VAL_RATIO)
    val_raw = train_csv_full[:n_val].copy()
    trn_raw = train_csv_full[n_val:].copy()
    if is_main:
        val_raw.to_csv(CKPT_DIR / "_val_raw.csv", index=False)
        print(f"[v18] Processes={accelerator.num_processes}  train={len(trn_raw)}  val={len(val_raw)}")

    model, processor = load_model_and_processor()
    yes_id, no_id = get_yes_no_token_ids(processor)

    train_ds = PairwiseTemporalDataset(trn_raw)
    n_steps = (len(trn_raw) // GRAD_ACCUM) * EPOCHS
    n_warmup = int(n_steps * WARMUP_RATIO)
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    scheduler = get_cosine_schedule_with_warmup(optimizer, n_warmup, n_steps)
    model, optimizer, scheduler = accelerator.prepare(model, optimizer, scheduler)

    history = []
    best_val_acc = 0.0
    global_step = 0
    step_times = []

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                               collate_fn=collate_fn, num_workers=4, pin_memory=True)
        train_dl = accelerator.prepare(train_dl)
        steps_per_epoch = len(trn_raw)
        if is_main:
            print(f"\n{'='*60}\nEpoch {epoch}/{EPOCHS}  (예상 step 수: {steps_per_epoch})\n{'='*60}")

        model.train()
        epoch_loss, n_batches = 0.0, 0
        t_step0 = time.time()
        for step, batch in enumerate(train_dl):
            with accelerator.accumulate(model):
                unwrapped = accelerator.unwrap_model(model)
                logit_parts, labels, adj_flags = [], [], []
                for sid, grp_imgs, grp_sents, grp_labels, grp_adj in zip(
                        batch["sids"], batch["images"], batch["sentences"], batch["labels"], batch["adj_flags"]):
                    base_imgs, sentence = grp_imgs[0], grp_sents[0]
                    z = _pairwise_logits_fast(unwrapped, processor, base_imgs, sentence, device,
                                               cached_vision=load_vision_cache(sid))
                    logit_parts.append(z)
                    labels.extend(grp_labels)
                    adj_flags.extend(grp_adj)
                logits = torch.cat(logit_parts)

                loss = pairwise_bt_loss(logits, labels, adj_flags)
                if not torch.isfinite(loss):
                    optimizer.zero_grad()
                    continue
                accelerator.backward(loss)
                if accelerator.sync_gradients:
                    accelerator.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                global_step += 1
                epoch_loss += loss.item()
                n_batches += 1

                step_dt = time.time() - t_step0
                step_times.append(step_dt)
                if len(step_times) > 20:
                    step_times.pop(0)
                avg_step = sum(step_times) / len(step_times)
                remaining = steps_per_epoch - (step + 1)
                eta_min = avg_step * remaining / 60
                cur_mem = torch.cuda.memory_allocated() / 1e9
                if is_main:
                    print(f"  step={global_step:5d} ({step+1}/{steps_per_epoch})  loss={epoch_loss/n_batches:.4f}  "
                          f"lr={scheduler.get_last_lr()[0]:.2e}  step_time={step_dt:.2f}s  avg={avg_step:.2f}s  "
                          f"ETA(에폭 잔여)={eta_min:.1f}분  mem={cur_mem:.1f}GB")
                t_step0 = time.time()

        accelerator.wait_for_everyone()
        if is_main:
            elapsed = (time.time() - t0) / 60
            avg_loss = epoch_loss / max(1, n_batches)
            print(f"\n[Epoch {epoch} 완료] {elapsed:.1f}분  avg_loss={avg_loss:.4f}")
            unwrapped = accelerator.unwrap_model(model)
            val_acc = val_exact_match(unwrapped, processor, val_raw, yes_id, no_id, device)
            history.append((epoch, avg_loss, val_acc))
            pd.DataFrame(history, columns=["epoch", "avg_loss", "val_acc"]).to_csv(LOG_DIR / "history.csv", index=False)
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                unwrapped.save_pretrained(CKPT_DIR / CKPT_NAME)
                processor.save_pretrained(CKPT_DIR / CKPT_NAME)
                print(f"  * Best 저장 (val_acc={val_acc:.4f})")
            unwrapped.save_pretrained(CKPT_DIR / (CKPT_NAME + "_last"))
            processor.save_pretrained(CKPT_DIR / (CKPT_NAME + "_last"))
        accelerator.wait_for_everyone()

    if is_main:
        print(f"\n학습 완료. Best val_acc={best_val_acc:.4f}")

print("train_fn 정의 완료")

## 11. 학습 실행

`notebook_launcher`가 `NUM_GPUS`개 프로세스를 fork해서 `train_fn`을 각 GPU에서 실행한다.
**§9 VRAM 자가진단(겸 스모크 테스트)을 먼저 돌려서 loss가 떨어지는지, `TRAIN_MINIBATCH`가
이 서버에서 안전한지 둘 다 확인 후 실행할 것.**

In [ ]:
from accelerate import notebook_launcher

notebook_launcher(train_fn, num_processes=NUM_GPUS, mixed_precision="bf16")

## 12. 학습 곡선 시각화

In [ ]:
import matplotlib.pyplot as plt

history_df = pd.read_csv(LOG_DIR / "history.csv")
print(history_df)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history_df["epoch"], history_df["avg_loss"], marker="o")
axes[0].set_title("avg_loss per epoch"); axes[0].set_xlabel("epoch")
axes[1].plot(history_df["epoch"], history_df["val_acc"], marker="o", color="orange")
axes[1].set_title("val exact_match per epoch"); axes[1].set_xlabel("epoch")
plt.tight_layout(); plt.show()

## 13. 추론 (샘플당 6-forward + 닫힌 형태 집계) + 제출 파일 생성

기존 v1~v17이 24 forward/샘플이었던 것과 비교하면 4배 절감. RTX 3090 단일 GPU 최종 실행
검증(`idea.md` §7 1순위 리스크)에도 직접적으로 도움이 되는 부분.

In [ ]:
def run_inference(ckpt_name, out_name):
    from peft import PeftModel
    from transformers import AutoProcessor
    base_model = get_model_class(MODEL_PATH).from_pretrained(
        MODEL_PATH, torch_dtype=torch.bfloat16, device_map={"": torch.device("cuda:0")}
    )
    processor = AutoProcessor.from_pretrained(MODEL_PATH)
    model = PeftModel.from_pretrained(base_model, str(CKPT_DIR / ckpt_name)).eval()
    yes_id, no_id = get_yes_no_token_ids(processor)

    test_df = pd.read_csv(DATA_DIR / "test.csv")
    submission = []
    for _, row in test_df.iterrows():
        sid, sentence = row["Id"], row["Sentence"]
        img_dir = DATA_DIR / "test" / sid
        files = sorted(f.name for f in img_dir.iterdir() if f.suffix == ".jpg")
        base_imgs = [load_image(str(img_dir / f)) for f in files]
        pred = predict_permutation(model, processor, yes_id, no_id, base_imgs, sentence, INFER_BATCH_SIZE)
        submission.append({"Id": sid, "Answer": str(pred)})

    out_path = PROJECT_ROOT / f"{out_name}.csv"
    pd.DataFrame(submission).to_csv(out_path, index=False)
    print(f"저장: {out_path} ({len(submission)}행)")
    del model, base_model
    torch.cuda.empty_cache()

run_inference(CKPT_NAME, "submission_v18_best")